In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import display

print("03 Recommendation Models - Started")

03 Recommendation Models - Started


In [2]:
PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
FEATURES_DIR = DATA_DIR / "features"
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"

FEATURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Features:", FEATURES_DIR)
print("Results:", RESULTS_DIR)

Project: c:\Users\ASUS\Documents\Smart-Tourism-Recommendation
Features: c:\Users\ASUS\Documents\Smart-Tourism-Recommendation\data\features
Results: c:\Users\ASUS\Documents\Smart-Tourism-Recommendation\results


In [3]:
destinations = pd.read_csv(
    PROCESSED_DIR / "destinations_clean.csv"
)

users = pd.read_csv(
    PROCESSED_DIR / "user_profiles_clean.csv"
)

evaluation_cases = pd.read_csv(
    PROCESSED_DIR / "evaluation_cases_clean.csv"
)

print("Datasets loaded")

print("Destinations:", destinations.shape)
print("Users:", users.shape)
print("Evaluation cases:", evaluation_cases.shape)

Datasets loaded
Destinations: (100, 21)
Users: (50, 13)
Evaluation cases: (500, 27)


In [4]:
destination_features = pd.read_csv(
    FEATURES_DIR / "destination_features.csv"
)

user_features = pd.read_csv(
    FEATURES_DIR / "user_features.csv"
)

evaluation_features = pd.read_csv(
    FEATURES_DIR / "evaluation_features.csv"
)

destination_text = pd.read_csv(
    FEATURES_DIR / "destination_text.csv"
)

user_text = pd.read_csv(
    FEATURES_DIR / "user_text.csv"
)

tfidf = joblib.load(
    FEATURES_DIR / "destination_tfidf_vectorizer.joblib"
)

print("Feature files loaded successfully")

print("\nDestination features:", destination_features.shape)
print("User features:", user_features.shape)
print("Evaluation features:", evaluation_features.shape)
print("Destination text:", destination_text.shape)
print("User text:", user_text.shape)

Feature files loaded successfully

Destination features: (100, 50)
User features: (50, 13)
Evaluation features: (500, 17)
Destination text: (100, 3)
User text: (50, 2)


In [5]:
print("========== DESTINATION FEATURES ==========")
display(destination_features.head())

print("\n========== USER FEATURES ==========")
display(user_features.head())

print("\n========== DESTINATION TEXT ==========")
display(destination_text.head())

print("\n========== USER TEXT ==========")
display(user_text.head())

========== DESTINATION FEATURES ==========


,destination_id,name,district,destination_text,estimated_cost_lkr_normalized,duration_hours_normalized,sustainability_score_normalized,crowd_score_normalized,seasonality_score_normalized,sentiment_nature_normalized,...,interest_nature,interest_photography,interest_rafting,interest_relaxation,interest_religious,interest_shopping,interest_snorkeling,interest_surfing,interest_water_sports,interest_wildlife
0,D001,Sigiriya Rock Fortress,matale,sigiriya rock fortress matale heritage nature ...,0.714286,0.166667,0.285714,0.833333,0.523810,0.257396,...,0,1,0,0,0,0,0,0,0,0
1,D002,Dambulla Cave Temple,matale,dambulla cave temple matale heritage heritage ...,0.214286,0.000000,0.190476,0.833333,0.000000,0.340237,...,0,1,0,0,0,0,0,0,0,0
2,D003,Minneriya National Park,polonnaruwa,minneriya national park polonnaruwa wildlife n...,0.928571,0.333333,0.904762,0.583333,0.571429,0.393491,...,1,1,0,0,0,0,0,0,0,1
3,D004,Kaudulla National Park,polonnaruwa,kaudulla national park polonnaruwa wildlife na...,0.928571,0.333333,0.619048,0.500000,0.904762,0.020710,...,1,1,0,0,0,0,0,0,0,1
4,D005,Polonnaruwa Ancient City,polonnaruwa,polonnaruwa ancient city polonnaruwa heritage ...,0.214286,0.333333,0.333333,0.458333,0.714286,0.511834,...,0,1,0,0,0,0,0,0,0,0



========== USER FEATURES ==========


,user_id,age_group,traveller_type,travel_style,starting_location,preferred_interests,preferred_crowd,preferred_weather,user_preference_text,budget_lkr_per_day_normalized,trip_duration_days_normalized,sustainability_importance_normalized,num_preferred_interests_normalized
0,U001,18-24,solo,budget,colombo,nature;photography,low,cool,nature photography budget solo low cool colombo,0.0,0.00,0.000000,0.0
1,U002,25-34,couple,moderate,kandy,beach;relaxation,medium,warm,beach relaxation moderate couple medium warm k...,0.2,0.25,0.166667,0.0
2,U003,35-44,friends,budget,gampaha,heritage;history,low,dry,heritage history budget friends low dry gampaha,0.4,0.50,0.333333,0.0
3,U004,18-24,family,moderate,negombo,wildlife;photography,medium,mixed,wildlife photography moderate family medium mi...,0.6,0.75,0.500000,0.0
4,U005,25-34,solo,premium,matara,adventure;hiking,high,rain_tolerant,adventure hiking premium solo high rain_tolera...,0.8,1.00,0.666667,0.0



========== DESTINATION TEXT ==========


,destination_id,name,destination_text
0,D001,Sigiriya Rock Fortress,sigiriya rock fortress matale heritage nature ...
1,D002,Dambulla Cave Temple,dambulla cave temple matale heritage heritage ...
2,D003,Minneriya National Park,minneriya national park polonnaruwa wildlife n...
3,D004,Kaudulla National Park,kaudulla national park polonnaruwa wildlife na...
4,D005,Polonnaruwa Ancient City,polonnaruwa ancient city polonnaruwa heritage ...



========== USER TEXT ==========


,user_id,user_preference_text
0,U001,nature photography budget solo low cool colombo
1,U002,beach relaxation moderate couple medium warm k...
2,U003,heritage history budget friends low dry gampaha
3,U004,wildlife photography moderate family medium mi...
4,U005,adventure hiking premium solo high rain_tolera...


In [6]:
destination_model = destinations.copy()

print(
    "Destination model dataset:",
    destination_model.shape
)

display(
    destination_model[
        [
            "destination_id",
            "name",
            "district",
            "category",
            "estimated_cost_lkr",
            "duration_hours",
            "interest_tags",
            "sustainability_score",
            "crowd_score",
            "seasonality_score",
            "weather_suitability"
        ]
    ].head()
)

Destination model dataset: (100, 21)


,destination_id,name,district,category,estimated_cost_lkr,duration_hours,interest_tags,sustainability_score,crowd_score,seasonality_score,weather_suitability
0,D001,Sigiriya Rock Fortress,matale,heritage;nature,12000,3.0,heritage;history;photography;hiking,57,41,58,0.516
1,D002,Dambulla Cave Temple,matale,heritage,5000,2.0,heritage;history;culture;photography,55,41,47,0.754
2,D003,Minneriya National Park,polonnaruwa,wildlife;nature,15000,4.0,wildlife;photography;nature;adventure,70,35,59,0.382
3,D004,Kaudulla National Park,polonnaruwa,wildlife;nature,15000,4.0,wildlife;photography;nature;adventure,64,33,66,0.366
4,D005,Polonnaruwa Ancient City,polonnaruwa,heritage,5000,4.0,heritage;history;culture;photography,58,32,62,0.467


In [7]:
def minmax_series(series):
    minimum = series.min()
    maximum = series.max()

    if maximum == minimum:
        return pd.Series(
            np.ones(len(series)),
            index=series.index
        )

    return (series - minimum) / (maximum - minimum)


destination_model["cost_normalized"] = (
    minmax_series(
        destination_model["estimated_cost_lkr"]
    )
)

destination_model["duration_normalized"] = (
    minmax_series(
        destination_model["duration_hours"]
    )
)

destination_model["sustainability_normalized"] = (
    destination_model["sustainability_score"] / 100
)

destination_model["crowd_normalized"] = (
    destination_model["crowd_score"] / 100
)

destination_model["seasonality_normalized"] = (
    destination_model["seasonality_score"] / 100
)

destination_model["weather_normalized"] = (
    destination_model["weather_suitability"]
)

destination_model["sentiment_normalized"] = (
    (
        destination_model["sentiment_nature"]
        + destination_model["sentiment_service"]
        + destination_model["sentiment_value"]
    ) / 3
)

print("Destination normalization completed.")

Destination normalization completed.


In [8]:
destination_model["baseline_score"] = (
    0.25 * destination_model["sustainability_normalized"]
    + 0.20 * destination_model["seasonality_normalized"]
    + 0.20 * destination_model["weather_normalized"]
    + 0.35 * destination_model["sentiment_normalized"]
)

print("Baseline scores calculated.")

display(
    destination_model[
        [
            "destination_id",
            "name",
            "baseline_score"
        ]
    ]
    .sort_values(
        "baseline_score",
        ascending=False
    )
    .head(10)
)

Baseline scores calculated.


,destination_id,name,baseline_score
56,D057,Kalpitiya Lagoon,0.757633
16,D017,Udawattakele Forest Reserve,0.756267
9,D010,Jaffna Fort,0.745150
30,D031,Nine Arches Bridge,0.737150
75,D076,Jungle Beach,0.733333
86,D087,Kirinda Temple,0.731533
96,D097,Haputale,0.729600
99,D100,Pekoe Trail - Tea Country Section,0.727083
82,D083,Hummanaya Blow Hole,0.724633
34,D035,Diyaluma Falls,0.724333


In [9]:
destination_model["baseline_score"] = (
    0.25 * destination_model["sustainability_normalized"]
    + 0.20 * destination_model["seasonality_normalized"]
    + 0.20 * destination_model["weather_normalized"]
    + 0.35 * destination_model["sentiment_normalized"]
)

print("Baseline scores calculated.")

display(
    destination_model[
        [
            "destination_id",
            "name",
            "baseline_score"
        ]
    ]
    .sort_values(
        "baseline_score",
        ascending=False
    )
    .head(10)
)

Baseline scores calculated.


,destination_id,name,baseline_score
56,D057,Kalpitiya Lagoon,0.757633
16,D017,Udawattakele Forest Reserve,0.756267
9,D010,Jaffna Fort,0.745150
30,D031,Nine Arches Bridge,0.737150
75,D076,Jungle Beach,0.733333
86,D087,Kirinda Temple,0.731533
96,D097,Haputale,0.729600
99,D100,Pekoe Trail - Tea Country Section,0.727083
82,D083,Hummanaya Blow Hole,0.724633
34,D035,Diyaluma Falls,0.724333


In [10]:
b1_rows = []

for _, user in users.iterrows():

    ranked = destination_model.sort_values(
        "baseline_score",
        ascending=False
    ).head(10)

    for rank, (_, destination) in enumerate(
        ranked.iterrows(),
        start=1
    ):

        b1_rows.append({
            "user_id": user["user_id"],
            "destination_id": destination["destination_id"],
            "name": destination["name"],
            "category": destination["category"],
            "district": destination["district"],
            "rank": rank,
            "baseline_score": destination["baseline_score"]
        })

b1_results = pd.DataFrame(b1_rows)

print("B1 recommendations:")
print(b1_results.shape)

display(b1_results.head(10))

B1 recommendations:
(500, 7)


,user_id,destination_id,name,category,district,rank,baseline_score
0,U001,D057,Kalpitiya Lagoon,marine;nature,puttalam,1,0.757633
1,U001,D017,Udawattakele Forest Reserve,nature,kandy,2,0.756267
2,U001,D010,Jaffna Fort,heritage,jaffna,3,0.745150
3,U001,D031,Nine Arches Bridge,heritage;nature,badulla,4,0.737150
4,U001,D076,Jungle Beach,beach;nature,galle,5,0.733333
5,U001,D087,Kirinda Temple,heritage;coastal,hambantota,6,0.731533
6,U001,D097,Haputale,nature;city,badulla,7,0.729600
7,U001,D100,Pekoe Trail - Tea Country Section,nature;adventure;culture,nuwara eliya,8,0.727083
8,U001,D083,Hummanaya Blow Hole,nature;coastal,matara,9,0.724633
9,U001,D035,Diyaluma Falls,nature;adventure,badulla,10,0.724333


In [11]:
b1_path = RESULTS_DIR / "b1_general_baseline_all_users_top10.csv"

b1_results.to_csv(
    b1_path,
    index=False
)

print("B1 saved:")
print(b1_path)

B1 saved:
c:\Users\ASUS\Documents\Smart-Tourism-Recommendation\results\b1_general_baseline_all_users_top10.csv


In [12]:
destination_text_matrix = tfidf.transform(
    destination_text["destination_text"].fillna("")
)

print(
    "Destination TF-IDF matrix:",
    destination_text_matrix.shape
)

Destination TF-IDF matrix: (100, 500)


In [13]:
def content_based_recommend(
    user_id,
    top_k=10
):

    user_row = user_text[
        user_text["user_id"] == user_id
    ]

    if user_row.empty:
        raise ValueError(
            f"User {user_id} not found."
        )

    user_text_value = str(
        user_row.iloc[0]["user_preference_text"]
    )

    user_vector = tfidf.transform(
        [user_text_value]
    )

    similarities = cosine_similarity(
        user_vector,
        destination_text_matrix
    )[0]

    result = destination_model.copy()

    result["content_similarity"] = similarities

    result = result.sort_values(
        "content_similarity",
        ascending=False
    ).head(top_k).copy()

    result["user_id"] = user_id

    result["rank"] = range(
        1,
        len(result) + 1
    )

    return result[
        [
            "user_id",
            "destination_id",
            "name",
            "category",
            "district",
            "content_similarity",
            "rank"
        ]
    ]

In [14]:
test_user_id = users["user_id"].iloc[0]

b2_test = content_based_recommend(
    test_user_id,
    top_k=10
)

print(
    "B2 test user:",
    test_user_id
)

display(b2_test)

B2 test user: U001


,user_id,destination_id,name,category,district,content_similarity,rank
52,U001,D053,Dutch Hospital Colombo,heritage;city,colombo,0.284874,1
50,U001,D051,Viharamahadevi Park,nature;city,colombo,0.284257,2
47,U001,D048,Colombo National Museum,heritage;culture,colombo,0.283490,3
51,U001,D052,Independence Square,heritage;city,colombo,0.223960,4
33,U001,D034,Lipton's Seat,nature,badulla,0.215905,5
92,U001,D093,Riverston,nature;adventure,matale,0.214148,6
27,U001,D028,Pidurutalagala View Area,nature,nuwara eliya,0.203146,7
49,U001,D050,Gangaramaya Temple,heritage;religious,colombo,0.201359,8
31,U001,D032,Ravana Falls,nature,badulla,0.197865,9
26,U001,D027,Ramboda Falls,nature,nuwara eliya,0.182088,10


In [15]:
b2_all = []

for user_id in users["user_id"]:

    result = content_based_recommend(
        user_id,
        top_k=10
    )

    b2_all.append(result)

all_b2_results = pd.concat(
    b2_all,
    ignore_index=True
)

print(
    "B2 all-user recommendations:",
    all_b2_results.shape
)

display(all_b2_results.head(10))

B2 all-user recommendations: (500, 7)


,user_id,destination_id,name,category,district,content_similarity,rank
0,U001,D053,Dutch Hospital Colombo,heritage;city,colombo,0.284874,1
1,U001,D051,Viharamahadevi Park,nature;city,colombo,0.284257,2
2,U001,D048,Colombo National Museum,heritage;culture,colombo,0.283490,3
3,U001,D052,Independence Square,heritage;city,colombo,0.223960,4
4,U001,D034,Lipton's Seat,nature,badulla,0.215905,5
5,U001,D093,Riverston,nature;adventure,matale,0.214148,6
6,U001,D028,Pidurutalagala View Area,nature,nuwara eliya,0.203146,7
7,U001,D050,Gangaramaya Temple,heritage;religious,colombo,0.201359,8
8,U001,D032,Ravana Falls,nature,badulla,0.197865,9
9,U001,D027,Ramboda Falls,nature,nuwara eliya,0.182088,10


In [16]:
b2_path = (
    RESULTS_DIR
    / "b2_content_based_all_users_top10.csv"
)

all_b2_results.to_csv(
    b2_path,
    index=False
)

print("B2 saved:")
print(b2_path)

B2 saved:
c:\Users\ASUS\Documents\Smart-Tourism-Recommendation\results\b2_content_based_all_users_top10.csv


In [17]:
def parse_interests(value):

    if pd.isna(value):
        return set()

    return {
        item.strip().lower()
        for item in str(value).split(";")
        if item.strip()
    }


def interest_match(
    user_interests,
    destination_interests
):

    user_set = parse_interests(
        user_interests
    )

    destination_set = parse_interests(
        destination_interests
    )

    if not user_set or not destination_set:
        return 0.0

    intersection = (
        user_set.intersection(
            destination_set
        )
    )

    return len(intersection) / len(
        user_set
    )

In [18]:
def calculate_budget_fit(
    user_budget,
    destination_cost
):

    if user_budget <= 0:
        return 0.0

    ratio = destination_cost / user_budget

    if ratio <= 1:
        return 1.0

    if ratio <= 1.5:
        return 1.0 - (
            (ratio - 1.0) / 0.5
        )

    return 0.0

In [19]:
def calculate_crowd_fit(
    preferred_crowd,
    crowd_score
):

    preferred_crowd = str(
        preferred_crowd
    ).lower()

    crowd_score = float(
        crowd_score
    )

    if preferred_crowd == "low":

        return max(
            0.0,
            1.0 - crowd_score / 50
        )

    if preferred_crowd == "medium":

        return max(
            0.0,
            1.0 - abs(crowd_score - 50) / 50
        )

    if preferred_crowd == "high":

        return crowd_score / 100

    return 0.5

In [20]:
def calculate_weather_fit(
    preferred_weather,
    destination_row
):

    preferred_weather = str(
        preferred_weather
    ).lower()

    weather_suitability = float(
        destination_row["weather_suitability"]
    )

    rain_risk = float(
        destination_row["weather_rain_risk"]
    )

    temperature = float(
        destination_row["weather_temp_c"]
    )

    if preferred_weather == "cool":

        temperature_fit = max(
            0.0,
            1.0 - abs(
                temperature - 22
            ) / 15
        )

        return (
            0.6 * temperature_fit
            + 0.4 * weather_suitability
        )

    if preferred_weather == "warm":

        temperature_fit = max(
            0.0,
            1.0 - abs(
                temperature - 29
            ) / 15
        )

        return (
            0.6 * temperature_fit
            + 0.4 * weather_suitability
        )

    if preferred_weather == "dry":

        return (
            0.7 * (1 - rain_risk)
            + 0.3 * weather_suitability
        )

    if preferred_weather == "rain_tolerant":

        return (
            0.5 * rain_risk
            + 0.5 * weather_suitability
        )

    return weather_suitability

In [21]:
def context_aware_recommend(
    user_id,
    top_k=10
):

    user_match = users[
        users["user_id"] == user_id
    ]

    if user_match.empty:
        raise ValueError(
            f"User {user_id} not found."
        )

    user = user_match.iloc[0]

    rows = []

    for _, destination in destination_model.iterrows():

        preference_score = interest_match(
            user["preferred_interests"],
            destination["interest_tags"]
        )

        budget_fit = calculate_budget_fit(
            user["budget_lkr_per_day"],
            destination["estimated_cost_lkr"]
        )

        crowd_fit = calculate_crowd_fit(
            user["preferred_crowd"],
            destination["crowd_score"]
        )

        weather_fit = calculate_weather_fit(
            user["preferred_weather"],
            destination
        )

        sustainability_fit = (
            destination["sustainability_normalized"]
            * float(
                user["sustainability_importance"]
            )
        )

        seasonality_fit = (
            destination["seasonality_normalized"]
        )

        sentiment_fit = (
            destination["sentiment_normalized"]
        )

        context_score = (
            0.30 * preference_score
            + 0.20 * budget_fit
            + 0.15 * crowd_fit
            + 0.10 * weather_fit
            + 0.10 * sustainability_fit
            + 0.05 * seasonality_fit
            + 0.10 * sentiment_fit
        )

        rows.append({

            "user_id": user_id,

            "destination_id":
                destination["destination_id"],

            "name":
                destination["name"],

            "category":
                destination["category"],

            "district":
                destination["district"],

            "preference_score":
                preference_score,

            "budget_fit":
                budget_fit,

            "crowd_fit":
                crowd_fit,

            "weather_fit":
                weather_fit,

            "sustainability_fit":
                sustainability_fit,

            "seasonality_fit":
                seasonality_fit,

            "sentiment_fit":
                sentiment_fit,

            "context_score":
                context_score
        })

    result = pd.DataFrame(rows)

    result = result.sort_values(
        "context_score",
        ascending=False
    ).head(top_k).copy()

    result["rank"] = range(
        1,
        len(result) + 1
    )

    return result

In [22]:
b3_test = context_aware_recommend(
    users["user_id"].iloc[0],
    top_k=10
)

display(b3_test)

,user_id,destination_id,name,category,district,preference_score,budget_fit,crowd_fit,weather_fit,sustainability_fit,seasonality_fit,sentiment_fit,context_score,rank
16,U001,D017,Udawattakele Forest Reserve,nature,kandy,1.0,1.0,0.52,0.7176,0.288,0.67,0.861333,0.798193,1
34,U001,D035,Diyaluma Falls,nature;adventure,badulla,1.0,1.0,0.48,0.8012,0.268,0.62,0.760667,0.785987,2
96,U001,D097,Haputale,nature;city,badulla,1.0,1.0,0.44,0.8104,0.244,0.52,0.874000,0.784840,3
23,U001,D024,Hakgala Botanical Garden,nature,nuwara eliya,1.0,1.0,0.38,0.8544,0.248,0.57,0.728000,0.768540,4
24,U001,D025,Ambewela Farm,nature;family,nuwara eliya,1.0,1.0,0.46,0.7260,0.232,0.61,0.732333,0.768533,5
14,U001,D015,Kandy Lake,nature;city,kandy,1.0,1.0,0.50,0.5904,0.252,0.53,0.824667,0.768207,6
25,U001,D026,Lover's Leap Waterfall,nature,nuwara eliya,1.0,1.0,0.48,0.5788,0.240,0.63,0.779000,0.763280,7
82,U001,D083,Hummanaya Blow Hole,nature;coastal,matara,1.0,1.0,0.32,0.7748,0.232,0.67,0.800667,0.762247,8
66,U001,D067,Bambarakanda Falls,nature,badulla,1.0,1.0,0.44,0.6228,0.248,0.53,0.778667,0.757447,9
83,U001,D084,Rekawa Turtle Beach,beach;wildlife,hambantota,1.0,1.0,0.30,0.6740,0.264,0.60,0.885333,0.757333,10


In [23]:
b3_all = []

for user_id in users["user_id"]:

    result = context_aware_recommend(
        user_id,
        top_k=10
    )

    b3_all.append(result)

all_b3_results = pd.concat(
    b3_all,
    ignore_index=True
)

print(
    "B3 all-user recommendations:",
    all_b3_results.shape
)

display(
    all_b3_results.head(10)
)

B3 all-user recommendations: (500, 14)


,user_id,destination_id,name,category,district,preference_score,budget_fit,crowd_fit,weather_fit,sustainability_fit,seasonality_fit,sentiment_fit,context_score,rank
0,U001,D017,Udawattakele Forest Reserve,nature,kandy,1.0,1.0,0.52,0.7176,0.288,0.67,0.861333,0.798193,1
1,U001,D035,Diyaluma Falls,nature;adventure,badulla,1.0,1.0,0.48,0.8012,0.268,0.62,0.760667,0.785987,2
2,U001,D097,Haputale,nature;city,badulla,1.0,1.0,0.44,0.8104,0.244,0.52,0.874000,0.784840,3
3,U001,D024,Hakgala Botanical Garden,nature,nuwara eliya,1.0,1.0,0.38,0.8544,0.248,0.57,0.728000,0.768540,4
4,U001,D025,Ambewela Farm,nature;family,nuwara eliya,1.0,1.0,0.46,0.7260,0.232,0.61,0.732333,0.768533,5
5,U001,D015,Kandy Lake,nature;city,kandy,1.0,1.0,0.50,0.5904,0.252,0.53,0.824667,0.768207,6
6,U001,D026,Lover's Leap Waterfall,nature,nuwara eliya,1.0,1.0,0.48,0.5788,0.240,0.63,0.779000,0.763280,7
7,U001,D083,Hummanaya Blow Hole,nature;coastal,matara,1.0,1.0,0.32,0.7748,0.232,0.67,0.800667,0.762247,8
8,U001,D067,Bambarakanda Falls,nature,badulla,1.0,1.0,0.44,0.6228,0.248,0.53,0.778667,0.757447,9
9,U001,D084,Rekawa Turtle Beach,beach;wildlife,hambantota,1.0,1.0,0.30,0.6740,0.264,0.60,0.885333,0.757333,10


In [24]:
b3_path = (
    RESULTS_DIR
    / "b3_context_aware_all_users_top10.csv"
)

all_b3_results.to_csv(
    b3_path,
    index=False
)

print("B3 saved:")
print(b3_path)


B3 saved:
c:\Users\ASUS\Documents\Smart-Tourism-Recommendation\results\b3_context_aware_all_users_top10.csv


In [25]:
all_b2_results["content_similarity_normalized"] = (
    all_b2_results
    .groupby("user_id")["content_similarity"]
    .transform(
        lambda x:
            (x - x.min()) /
            (x.max() - x.min())
            if x.max() != x.min()
            else 1.0
    )
)

print("B2 scores normalized.")

B2 scores normalized.


In [26]:
b3_for_hybrid = all_b3_results[
    [
        "user_id",
        "destination_id",
        "context_score"
    ]
].copy()

print(
    "B3 hybrid data:",
    b3_for_hybrid.shape
)

B3 hybrid data: (500, 3)


In [27]:
hybrid_candidates = pd.merge(
    all_b2_results[
        [
            "user_id",
            "destination_id",
            "content_similarity",
            "content_similarity_normalized"
        ]
    ],

    b3_for_hybrid,

    on=[
        "user_id",
        "destination_id"
    ],

    how="inner"
)

print(
    "Hybrid candidate rows:",
    hybrid_candidates.shape
)

display(
    hybrid_candidates.head()
)

Hybrid candidate rows: (206, 5)


,user_id,destination_id,content_similarity,content_similarity_normalized,context_score
0,U002,D088,0.499895,1.000000,0.832527
1,U002,D041,0.444962,0.700368,0.815013
2,U002,D047,0.437344,0.658816,0.822867
3,U002,D012,0.428244,0.609184,0.857480
4,U002,D079,0.404958,0.482170,0.808407


In [28]:
hybrid_candidates["hybrid_score"] = (
    0.60
    * hybrid_candidates[
        "content_similarity_normalized"
    ]
    +
    0.40
    * hybrid_candidates[
        "context_score"
    ]
)

print("Hybrid scores calculated.")

display(
    hybrid_candidates.sort_values(
        ["user_id", "hybrid_score"],
        ascending=[True, False]
    ).head(10)
)

Hybrid scores calculated.


,user_id,destination_id,content_similarity,content_similarity_normalized,context_score,hybrid_score
0,U002,D088,0.499895,1.000000,0.832527,0.933011
1,U002,D041,0.444962,0.700368,0.815013,0.746226
2,U002,D047,0.437344,0.658816,0.822867,0.724436
3,U002,D012,0.428244,0.609184,0.857480,0.708502
4,U002,D079,0.404958,0.482170,0.808407,0.612665
5,U002,D055,0.402058,0.466353,0.798707,0.599294
6,U002,D054,0.380438,0.348428,0.826373,0.539606
7,U002,D075,0.316558,0.000000,0.846547,0.338619
8,U003,D037,0.323036,1.000000,0.758587,0.903435
9,U003,D008,0.276532,0.248374,0.744893,0.446982


In [29]:
all_hybrid_results = (
    hybrid_candidates
    .sort_values(
        ["user_id", "hybrid_score"],
        ascending=[True, False]
    )
    .groupby("user_id")
    .head(10)
    .copy()
)

all_hybrid_results["rank"] = (
    all_hybrid_results
    .groupby("user_id")
    .cumcount()
    + 1
)

print(
    "B4 hybrid recommendations:",
    all_hybrid_results.shape
)

display(
    all_hybrid_results.head(10)
)

B4 hybrid recommendations: (206, 7)


,user_id,destination_id,content_similarity,content_similarity_normalized,context_score,hybrid_score,rank
0,U002,D088,0.499895,1.000000,0.832527,0.933011,1
1,U002,D041,0.444962,0.700368,0.815013,0.746226,2
2,U002,D047,0.437344,0.658816,0.822867,0.724436,3
3,U002,D012,0.428244,0.609184,0.857480,0.708502,4
4,U002,D079,0.404958,0.482170,0.808407,0.612665,5
5,U002,D055,0.402058,0.466353,0.798707,0.599294,6
6,U002,D054,0.380438,0.348428,0.826373,0.539606,7
7,U002,D075,0.316558,0.000000,0.846547,0.338619,8
8,U003,D037,0.323036,1.000000,0.758587,0.903435,1
9,U003,D008,0.276532,0.248374,0.744893,0.446982,2


In [30]:
destination_details = destination_model[
    [
        "destination_id",
        "name",
        "category",
        "district",
        "estimated_cost_lkr",
        "duration_hours",
        "sustainability_score",
        "crowd_score",
        "seasonality_score",
        "weather_temp_c",
        "weather_rain_risk",
        "weather_suitability"
    ]
].copy()

all_hybrid_results = all_hybrid_results.merge(
    destination_details,
    on="destination_id",
    how="left"
)

print(
    "Hybrid results with destination details:",
    all_hybrid_results.shape
)

display(all_hybrid_results.head())


Hybrid results with destination details: (206, 18)


,user_id,destination_id,content_similarity,content_similarity_normalized,context_score,hybrid_score,rank,name,category,district,estimated_cost_lkr,duration_hours,sustainability_score,crowd_score,seasonality_score,weather_temp_c,weather_rain_risk,weather_suitability
0,U002,D088,0.499895,1.000000,0.832527,0.933011,1,Tangalle Beach,beach,hambantota,5000,4.0,55,37,61,28.8,0.061,0.939
1,U002,D041,0.444962,0.700368,0.815013,0.746226,2,Pasikuda Beach,beach,batticaloa,7000,4.0,58,35,51,27.2,0.249,0.751
2,U002,D047,0.437344,0.658816,0.822867,0.724436,3,Marble Beach,beach,trincomalee,5000,3.0,52,33,61,28.9,0.259,0.741
3,U002,D012,0.428244,0.609184,0.857480,0.708502,4,Casuarina Beach,beach,jaffna,4000,3.0,58,43,50,29.1,0.134,0.866
4,U002,D079,0.404958,0.482170,0.808407,0.612665,5,Mirissa Beach,beach,matara,6000,4.0,56,36,63,24.2,0.301,0.699


In [31]:
def create_hybrid_explanation(row):

    reasons = []

    if row["content_similarity_normalized"] >= 0.6:
        reasons.append(
            "strong preference match"
        )

    if row["context_score"] >= 0.6:
        reasons.append(
            "good contextual suitability"
        )

    if row["estimated_cost_lkr"] <= 5000:
        reasons.append(
            "fits a lower travel budget"
        )

    if row["sustainability_score"] >= 70:
        reasons.append(
            "good sustainability alignment"
        )

    if row["weather_suitability"] >= 0.75:
        reasons.append(
            "suitable weather conditions"
        )

    if not reasons:
        reasons.append(
            "overall recommendation score"
        )

    return "; ".join(reasons)


all_hybrid_results["explanation"] = (
    all_hybrid_results.apply(
        create_hybrid_explanation,
        axis=1
    )
)

display(
    all_hybrid_results[
        [
            "user_id",
            "destination_id",
            "name",
            "hybrid_score",
            "explanation"
        ]
    ].head(10)
)

,user_id,destination_id,name,hybrid_score,explanation
0,U002,D088,Tangalle Beach,0.933011,strong preference match; good contextual suita...
1,U002,D041,Pasikuda Beach,0.746226,strong preference match; good contextual suita...
2,U002,D047,Marble Beach,0.724436,strong preference match; good contextual suita...
3,U002,D012,Casuarina Beach,0.708502,strong preference match; good contextual suita...
4,U002,D079,Mirissa Beach,0.612665,good contextual suitability
5,U002,D055,Negombo Beach,0.599294,good contextual suitability; fits a lower trav...
6,U002,D054,Mount Lavinia Beach,0.539606,good contextual suitability; fits a lower trav...
7,U002,D075,Unawatuna Beach,0.338619,good contextual suitability
8,U003,D037,Buduruwagala,0.903435,strong preference match; good contextual suita...
9,U003,D008,Mihintale,0.446982,good contextual suitability; fits a lower trav...


In [32]:
b4_path = (
    RESULTS_DIR
    / "b4_hybrid_all_users_top10.csv"
)

all_hybrid_results.to_csv(
    b4_path,
    index=False
)

print("B4 saved:")
print(b4_path)

B4 saved:
c:\Users\ASUS\Documents\Smart-Tourism-Recommendation\results\b4_hybrid_all_users_top10.csv


In [33]:
hybrid_candidates_path = (
    RESULTS_DIR
    / "b4_hybrid_all_candidates.csv"
)

hybrid_candidates.to_csv(
    hybrid_candidates_path,
    index=False
)

print("All hybrid candidates saved:")
print(hybrid_candidates_path)

All hybrid candidates saved:
c:\Users\ASUS\Documents\Smart-Tourism-Recommendation\results\b4_hybrid_all_candidates.csv


In [34]:
test_user_id = users["user_id"].iloc[0]

print("TEST USER:", test_user_id)

display(
    all_hybrid_results[
        all_hybrid_results["user_id"] == test_user_id
    ][
        [
            "rank",
            "destination_id",
            "name",
            "category",
            "hybrid_score",
            "content_similarity_normalized",
            "context_score",
            "explanation"
        ]
    ]
)

TEST USER: U001


,rank,destination_id,name,category,hybrid_score,content_similarity_normalized,context_score,explanation


In [35]:
recommendation_counts = (
    all_hybrid_results
    .groupby("user_id")
    .size()
)

print("Number of users:")
print(recommendation_counts.shape[0])

print("\nRecommendations per user:")
print(recommendation_counts.value_counts().sort_index())

print(
    "\nTotal recommendations:",
    len(all_hybrid_results)
)

Number of users:
45

Recommendations per user:
1     2
2     6
3     9
4     6
5    12
7     3
8     3
9     4
Name: count, dtype: int64

Total recommendations: 206


In [36]:
print("========== SCORE RANGES ==========")

print(
    "Content similarity:",
    all_b2_results["content_similarity"].min(),
    "to",
    all_b2_results["content_similarity"].max()
)

print(
    "Context score:",
    all_b3_results["context_score"].min(),
    "to",
    all_b3_results["context_score"].max()
)

print(
    "Hybrid score:",
    all_hybrid_results["hybrid_score"].min(),
    "to",
    all_hybrid_results["hybrid_score"].max()
)

========== SCORE RANGES ==========
Content similarity: 0.12402939036894889 to 0.4998953527623206
Context score: 0.6011 to 0.8837266666666667
Hybrid score: 0.24222666666666665 to 0.9418106666666666


In [37]:
print("=" * 70)
print("03 RECOMMENDATION MODELS COMPLETE")
print("=" * 70)

print("\nB1 Baseline:")
print(b1_results.shape)

print("\nB2 Content-Based:")
print(all_b2_results.shape)

print("\nB3 Context-Aware:")
print(all_b3_results.shape)

print("\nB4 Hybrid:")
print(all_hybrid_results.shape)

print("\nSaved result files:")

for file in sorted(RESULTS_DIR.iterdir()):
    print(" -", file.name)

print("\nAll recommendation models completed successfully.")

03 RECOMMENDATION MODELS COMPLETE

B1 Baseline:
(500, 7)

B2 Content-Based:
(500, 8)

B3 Context-Aware:
(500, 14)

B4 Hybrid:
(206, 19)

Saved result files:
 - actual_vs_predicted_crowd.png
 - b1_general_baseline_all_users_top10.csv
 - b2_content_based_all_users_top10.csv
 - b2_content_based_recommendations.csv
 - b3_context_aware_all_users_top10.csv
 - b3_context_aware_recommendations.csv
 - b4_hybrid_all_candidates.csv
 - b4_hybrid_all_users_top10.csv
 - b4_hybrid_explainable_recommendations.csv
 - b4_hybrid_recommendations.csv
 - b4_vs_demand_aware_comparison.csv
 - b4_vs_demand_aware_comparison.png
 - crowd_prediction_feature_importance.png
 - crowd_prediction_metrics.csv
 - d1_sbert_semantic_recommendations.csv
 - destination_crowd_predictions.csv
 - final_demand_aware_recommendations.csv
 - final_model_weights.csv
 - model_comparison_metrics.csv
 - model_comparison_metrics.png
 - tfidf_vs_sbert_comparison.csv

All recommendation models completed successfully.
